In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.callbacks import Callback
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings("ignore")
np.random.seed(42)
tf.random.set_seed(42)

# Load Fashion-MNIST
print("Loading Fashion-MNIST dataset...")
(X_all, y_all), (X_test_raw, y_test) = fashion_mnist.load_data()

# Normalize images
X_all = X_all.astype("float32") / 255.0
X_test = X_test_raw.astype("float32") / 255.0

# Flatten images for feedforward networks
X_all = X_all.reshape(len(X_all), 784)
X_test = X_test.reshape(len(X_test), 784)

# Use a subset to make experiments faster
X_train, X_val, y_train, y_val = train_test_split(
    X_all[:20000], y_all[:20000],
    test_size=0.2, random_state=42, stratify=y_all[:20000]
)

X_test = X_test[:5000]
y_test = y_test[:5000]

NUM_CLASSES = 10
BATCH_SIZE = 256
EPOCHS = 5

# Model builder
def build_model(hidden_units=(256, 128), activation="relu",
                learning_rate=0.001, optimizer_name="adam",
                dropout=0.0):
    model = models.Sequential()
    model.add(layers.Input(shape=(784,)))

    for units in hidden_units:
        if activation == "leaky_relu":
            model.add(layers.Dense(units))
            model.add(layers.LeakyReLU(negative_slope=0.01))
        elif activation == "elu":
            model.add(layers.Dense(units, activation="elu"))
        else:
            model.add(layers.Dense(units, activation=activation))

        if dropout > 0:
            model.add(layers.Dropout(dropout))

    model.add(layers.Dense(NUM_CLASSES, activation="softmax"))

    if optimizer_name == "sgd":
        optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    else:
        optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# Callback to monitor loss, accuracy and gradient magnitudes
class TrainingMonitor(Callback):
    def __init__(self, sample_X, sample_y):
        super().__init__()
        self.sample_X = sample_X[:128]
        self.sample_y = sample_y[:128]
        self.gradient_history = []
        self.activation_history = []
        self.loss_history = []
        self.val_loss_history = []
        self.accuracy_history = []
        self.val_accuracy_history = []
        self.nan_detected = False

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        self.loss_history.append(logs.get("loss", np.nan))
        self.val_loss_history.append(logs.get("val_loss", np.nan))
        self.accuracy_history.append(logs.get("accuracy", np.nan))
        self.val_accuracy_history.append(logs.get("val_accuracy", np.nan))

        values = [
            logs.get("loss", np.nan),
            logs.get("val_loss", np.nan),
            logs.get("accuracy", np.nan),
            logs.get("val_accuracy", np.nan)
        ]
        if not np.all(np.isfinite(values)):
            self.nan_detected = True

        x = tf.convert_to_tensor(self.sample_X)
        y = tf.convert_to_tensor(self.sample_y)

        with tf.GradientTape() as tape:
            logits = self.model(x, training=True)
            loss = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(y, logits)
            )

        gradients = tape.gradient(loss, self.model.trainable_variables)
        layer_gradients = []
        for layer in self.model.layers:
            layer_grads = [
                tf.reduce_mean(tf.abs(g)).numpy()
                for variable, g in zip(
                    self.model.trainable_variables, gradients
                )
                if g is not None and variable.shape == layer.weights[0].shape
            ] if layer.weights else []

            if layer_grads:
                layer_gradients.append(float(np.mean(layer_grads)))

        self.gradient_history.append(layer_gradients)

# Utility functions
def gradient_report(monitor):
    if not monitor.gradient_history:
        return "Unavailable", []

    latest = monitor.gradient_history[-1]
    if not latest:
        return "Unavailable", []

    diagnosis = []
    for i, magnitude in enumerate(latest, 1):
        status = "Vanishing" if magnitude < 1e-7 else "Normal"
        diagnosis.append({
            "Layer": i,
            "Average Gradient": magnitude,
            "Status": status
        })

    if any(x["Status"] == "Vanishing" for x in diagnosis):
        overall = "Vanishing Gradient"
    else:
        overall = "Normal Gradient"

    return overall, diagnosis

def get_activations(model, X, layer_index):
    activation_model = tf.keras.Model(
        inputs=model.input,
        outputs=model.layers[layer_index].output
    )
    return activation_model.predict(X, verbose=0)

def dead_relu_report(model, X):
    records = []
    for i, layer in enumerate(model.layers):
        if isinstance(layer, layers.Dense) and i < len(model.layers) - 1:
            activations = get_activations(model, X, i)
            dead = np.all(activations <= 0, axis=0)
            records.append({
                "Layer": layer.name,
                "Neurons": activations.shape[1],
                "Dead Neurons": int(dead.sum()),
                "Dead Percentage (%)": float(dead.mean() * 100)
            })
    return records

def train_experiment(name, hidden_units, activation, learning_rate=0.001,
                     epochs=EPOCHS, optimizer_name="adam", dropout=0.0):
    tf.keras.backend.clear_session()
    net = build_model(
        hidden_units=hidden_units,
        activation=activation,
        learning_rate=learning_rate,
        optimizer_name=optimizer_name,
        dropout=dropout
    )

    monitor = TrainingMonitor(X_train, y_train)
    history = net.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=BATCH_SIZE,
        verbose=0,
        callbacks=[monitor]
    )

    train_loss, train_acc = net.evaluate(X_train, y_train, verbose=0)
    val_loss, val_acc = net.evaluate(X_val, y_val, verbose=0)
    test_loss, test_acc = net.evaluate(X_test, y_test, verbose=0)

    gradient_status, gradient_details = gradient_report(monitor)
    dead_records = dead_relu_report(net, X_val)

    increasing_loss = (
        len(monitor.loss_history) >= 3 and
        all(
            monitor.loss_history[i] > monitor.loss_history[i - 1]
            for i in range(1, len(monitor.loss_history))
        )
    )

    return {
        "name": name,
        "model": net,
        "history": history.history,
        "monitor": monitor,
        "train_accuracy": train_acc * 100,
        "val_accuracy": val_acc * 100,
        "test_accuracy": test_acc * 100,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "test_loss": test_loss,
        "gradient_status": gradient_status,
        "gradient_details": gradient_details,
        "dead_records": dead_records,
        "increasing_loss": increasing_loss
    }

# 1. Extremely Low and High Learning Rate
def learning_rate_analyzer():
    results = []
    rates = [0.0000001, 0.001, 0.1, 1.0, 10.0]

    for lr in rates:
        print(f"\nTraining with learning rate {lr}...")
        try:
            result = train_experiment(
                f"LR={lr}",
                hidden_units=(128, 64),
                activation="relu",
                learning_rate=lr,
                epochs=3,
                optimizer_name="sgd"
            )

            losses = np.array(result["monitor"].loss_history)
            if result["monitor"].nan_detected or not np.all(np.isfinite(losses)):
                behaviour = "Unstable"
            elif len(losses) >= 2 and losses[-1] > losses[0] * 1.5:
                behaviour = "Unstable"
            elif lr == 0.0000001 or (
                len(losses) >= 2 and abs(losses[0] - losses[-1]) < 0.001
            ):
                behaviour = "Too Slow"
            else:
                behaviour = "Stable"

            results.append({
                "Learning Rate": lr,
                "Training Loss": np.round(losses, 4).tolist(),
                "Training Accuracy (%)": result["train_accuracy"],
                "Validation Accuracy (%)": result["val_accuracy"],
                "Behaviour": behaviour,
                "NaN/Inf Detected": result["monitor"].nan_detected
            })

        except Exception as e:
            results.append({
                "Learning Rate": lr,
                "Training Loss": str(e),
                "Training Accuracy (%)": np.nan,
                "Validation Accuracy (%)": np.nan,
                "Behaviour": "Unstable",
                "NaN/Inf Detected": True
            })

    print("\nLEARNING RATE BEHAVIOUR REPORT")
    print(pd.DataFrame(results).to_string(index=False))

# 2. Vanishing Gradient Analyzer
def vanishing_gradient_analyzer():
    results = []

    for activation in ["sigmoid", "tanh", "relu"]:
        print(f"\nTraining DFNN using {activation}...")
        result = train_experiment(
            activation,
            hidden_units=(256, 192, 128, 96, 64, 32),
            activation=activation,
            learning_rate=0.001,
            epochs=3
        )

        results.append({
            "Activation": activation,
            "Train Accuracy (%)": result["train_accuracy"],
            "Validation Accuracy (%)": result["val_accuracy"],
            "Gradient Behaviour": result["gradient_status"],
            "Layer Gradients": result["gradient_details"]
        })

    print("\nVANISHING GRADIENT REPORT")
    print(pd.DataFrame(results).to_string(index=False))

# 3. Dead ReLU Neuron Detector
def dead_relu_detector():
    result = train_experiment(
        "ReLU MLP",
        hidden_units=(256, 128, 64),
        activation="relu",
        learning_rate=0.001,
        epochs=EPOCHS
    )

    print("\nDEAD RELU NEURON REPORT")
    print(pd.DataFrame(result["dead_records"]).to_string(index=False))

# 4. Overfitting Detector
def overfitting_detector():
    result = train_experiment(
        "High Capacity DFNN",
        hidden_units=(512, 512, 256, 128, 64),
        activation="relu",
        learning_rate=0.001,
        epochs=15,
        dropout=0.0
    )

    hist = result["history"]
    overfit_epoch = None
    streak = 0

    for i in range(1, len(hist["accuracy"])):
        train_up = hist["accuracy"][i] > hist["accuracy"][i - 1]
        val_down = hist["val_accuracy"][i] < hist["val_accuracy"][i - 1]

        if train_up and val_down:
            streak += 1
        else:
            streak = 0

        if streak >= 3:
            overfit_epoch = i - 2 + 1
            break

    print("\nOVERFITTING DETECTOR")
    print("Epoch | Train Accuracy | Validation Accuracy | Train Loss | Validation Loss")
    for i in range(len(hist["accuracy"])):
        print(
            f"{i+1:5d} | {hist['accuracy'][i]:.4f} | "
            f"{hist['val_accuracy'][i]:.4f} | "
            f"{hist['loss'][i]:.4f} | {hist['val_loss'][i]:.4f}"
        )

    if overfit_epoch is not None:
        print("Possible overfitting first detected at epoch:", overfit_epoch)
    else:
        print("No sustained overfitting pattern detected under the specified rule.")

# 5. Activation Function Tournament
def activation_tournament():
    results = []

    for activation in ["relu", "sigmoid", "tanh", "leaky_relu", "elu"]:
        print(f"\nTraining model with {activation}...")
        result = train_experiment(
            activation,
            hidden_units=(128, 64),
            activation=activation,
            learning_rate=0.001,
            epochs=EPOCHS
        )

        results.append({
            "Activation": activation,
            "Train Accuracy (%)": result["train_accuracy"],
            "Validation Accuracy (%)": result["val_accuracy"],
            "Test Accuracy (%)": result["test_accuracy"],
            "Final Training Loss": result["train_loss"],
            "Final Validation Loss": result["val_loss"]
        })

    df = pd.DataFrame(results).sort_values(
        "Test Accuracy (%)", ascending=False
    )
    print("\nACTIVATION FUNCTION TOURNAMENT")
    print(df.round(4).to_string(index=False))

# 6. Automatic Failure Diagnosis
def failure_diagnosis():
    result = train_experiment(
        "Diagnostic FNN",
        hidden_units=(256, 128, 64),
        activation="relu",
        learning_rate=0.001,
        epochs=EPOCHS
    )

    hist = result["history"]
    dead_percentage = max(
        [r["Dead Percentage (%)"] for r in result["dead_records"]],
        default=0
    )

    train_acc = np.array(hist["accuracy"])
    val_acc = np.array(hist["val_accuracy"])
    val_loss = np.array(hist["val_loss"])

    if result["monitor"].nan_detected or result["increasing_loss"]:
        diagnosis = "Learning Rate Too High"
        recommendation = "Reduce the learning rate and use gradient clipping."
        metric = "Non-finite or increasing training loss"
    elif result["gradient_status"] == "Vanishing Gradient":
        diagnosis = "Vanishing Gradient"
        recommendation = "Use ReLU, He initialization, or residual connections."
        metric = result["gradient_details"]
    elif dead_percentage > 30:
        diagnosis = "Dead ReLU Neurons"
        recommendation = "Try LeakyReLU or ELU and reduce the learning rate."
        metric = f"Maximum dead-neuron percentage: {dead_percentage:.2f}%"
    elif len(val_acc) >= 3 and train_acc[-1] > val_acc[-1] + 0.10:
        diagnosis = "Possible Overfitting"
        recommendation = "Add dropout, regularization, or early stopping."
        metric = f"Train accuracy {train_acc[-1]:.4f}, validation {val_acc[-1]:.4f}"
    elif len(hist["loss"]) >= 2 and hist["loss"][-1] >= hist["loss"][0] * 0.95:
        diagnosis = "Slow Convergence"
        recommendation = "Try a higher learning rate or an adaptive optimizer."
        metric = f"Loss: {hist['loss'][0]:.4f} -> {hist['loss'][-1]:.4f}"
    else:
        diagnosis = "Stable Training"
        recommendation = "Continue monitoring validation performance."
        metric = (
            f"Train accuracy: {result['train_accuracy']:.2f}%, "
            f"validation accuracy: {result['val_accuracy']:.2f}%"
        )

    print("\nAUTOMATIC FAILURE DIAGNOSIS")
    print("Detected Problem:", diagnosis)
    print("Supporting Metric:", metric)
    print("Recommendation:", recommendation)

# 7. Comparative Configuration Analyzer
def comparative_configuration_analyzer():
    configurations = [
        ("FNN", (128,), "relu", 0.001),
        ("MLP", (256, 128), "relu", 0.001),
        ("DFNN", (256, 192, 128, 96, 64, 32), "relu", 0.001),
        ("DFNN-Sigmoid", (256, 192, 128, 96, 64, 32), "sigmoid", 0.001),
        ("DFNN-Tanh", (256, 192, 128, 96, 64, 32), "tanh", 0.001)
    ]

    results = []

    for name, units, activation, lr in configurations:
        print(f"\nTraining {name}...")
        result = train_experiment(
            name,
            hidden_units=units,
            activation=activation,
            learning_rate=lr,
            epochs=EPOCHS
        )

        dead_percentage = np.mean([
            row["Dead Percentage (%)"]
            for row in result["dead_records"]
        ]) if result["dead_records"] else 0

        results.append({
            "Network": name,
            "Hidden Layers": len(units),
            "Activation": activation,
            "Learning Rate": lr,
            "Train Accuracy (%)": result["train_accuracy"],
            "Validation Accuracy (%)": result["val_accuracy"],
            "Test Accuracy (%)": result["test_accuracy"],
            "Gradient Behaviour": result["gradient_status"],
            "Dead Neuron (%)": dead_percentage,
            "Training Behaviour": (
                "Unstable" if result["monitor"].nan_detected
                else "Loss Increasing" if result["increasing_loss"]
                else "Stable/Needs Monitoring"
            )
        })

    df = pd.DataFrame(results).sort_values(
        "Validation Accuracy (%)", ascending=False
    )
    print("\nCOMPARATIVE CONFIGURATION REPORT")
    print(df.round(4).to_string(index=False))
    print("\nBest Configuration Based on Validation Accuracy:")
    print(df.iloc[0].to_string())

# Menu
def main_menu():
    while True:
        print("\n" + "=" * 70)
        print(" NEURAL NETWORK TRAINING FAILURE DIAGNOSIS SYSTEM")
        print("=" * 70)
        print("1. Test Extremely Low Learning Rate")
        print("2. Test Extremely High Learning Rate")
        print("3. Detect Vanishing Gradient")
        print("4. Detect Dead ReLU Neurons")
        print("5. Detect Overfitting")
        print("6. Test Stable Neural Network Configuration")
        print("7. Compare All Failure Cases")
        print("8. Exit")

        choice = input("\nEnter your choice (1-8): ").strip()

        if choice == "1":
            learning_rate_analyzer()
        elif choice == "2":
            learning_rate_analyzer()
        elif choice == "3":
            vanishing_gradient_analyzer()
        elif choice == "4":
            dead_relu_detector()
        elif choice == "5":
            overfitting_detector()
        elif choice == "6":
            result = train_experiment(
                "Stable Configuration",
                hidden_units=(128, 64),
                activation="relu",
                learning_rate=0.001,
                epochs=EPOCHS,
                dropout=0.2
            )
            print("\nSTABLE CONFIGURATION")
            print(f"Training Accuracy: {result['train_accuracy']:.2f}%")
            print(f"Validation Accuracy: {result['val_accuracy']:.2f}%")
            print(f"Test Accuracy: {result['test_accuracy']:.2f}%")
            print(f"Training Loss: {result['train_loss']:.4f}")
            print(f"Validation Loss: {result['val_loss']:.4f}")
            print("Gradient Behaviour:", result["gradient_status"])
        elif choice == "7":
            comparative_configuration_analyzer()
            failure_diagnosis()
            activation_tournament()
        elif choice == "8":
            print("\nPROGRAM FINISHED")
            break
        else:
            print("Invalid choice. Enter a number from 1 to 8.")

main_menu()

Loading Fashion-MNIST dataset...
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

 NEURAL NETWORK TRAINING FAILURE DIAGNOSIS SYSTEM
1. Test Extremely Low Learning Rate
2. Test Extremely High Learning Rate
3. Detect Vanishing Gradient
4. Detect Dead ReLU Neurons
5. Detect Overfitting
6. Test Stable Neural Network Configuration
7. Compare All Failure Cases
8. Exit

Enter your choice (1-8): 8
